# 第一课：交通数据如何变成监督学习样本

这一课不训练模型。我们的目标是亲手回答四个问题：

1. METR-LA 文件中的一行、一列和一个数分别代表什么？
2. 模型的输入 `x [B,T_in,N,F]` 和目标 `y [B,T_out,N]` 是怎样产生的？
3. 缺失值和标准化是怎样处理的？
4. 为什么按时间切分后，训练目标不会泄漏到验证集和测试集？

> 学习方法：看到“先思考”时先写下答案，再运行下一格。不要一次性 Run All。

## 0. 环境与路径

请先把项目虚拟环境注册成独立的 Jupyter kernel，再从项目根目录启动 Jupyter：

```bash
source .venv/bin/activate
python -m pip install ipykernel
python -m ipykernel install --user --name stmining --display-name 'Python (STmining .venv)'
jupyter lab --allow-root --no-browser --ip=0.0.0.0 --port=8889
```

打开 notebook 后，在右上角 kernel 菜单中选择 **Python (STmining .venv)**。Jupyter 网页服务器可以来自系统 Conda，但执行代码的 kernel 必须来自 `.venv`。下面的代码会自动寻找项目根目录，因此从 `notebooks/` 打开也能运行。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '请从项目根目录或 notebooks 目录启动 notebook'
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.config import load_config  # noqa: E402
from crosscity.data.dataset import build_data_bundle, load_speed_matrix  # noqa: E402

CONFIG_PATH = ROOT / 'configs/metr_la.yaml'
config = load_config(CONFIG_PATH)
config

配置中最重要的三个数是：`input_steps=12`、`output_steps=12`、`steps_per_day=288`。

METR-LA 每 5 分钟记录一次，所以一天有 $24\times60/5=288$ 个时间点。模型将读取过去 12 步，也就是 60 分钟，并预测未来 12 步。

## 1. 先看未经处理的速度矩阵

**先思考：** 你预计 METR-LA 的数组形状是什么？哪个维度应该等于 207？

In [ ]:
data_path = Path(config.dataset.data_path)
assert data_path.exists(), f'没有找到数据文件：{data_path}'
raw = load_speed_matrix(data_path)

print('数组形状 [时间点, 传感器]:', raw.shape)
print('时间点数量:', raw.shape[0])
print('传感器数量:', raw.shape[1])
print('大约包含多少天:', raw.shape[0] / config.dataset.steps_per_day)
print('数据类型:', raw.dtype)

矩阵 `raw[t, n]` 表示：第 `t` 个五分钟时刻，编号为 `n` 的传感器记录到的交通速度。

这里的节点不是一辆车，也不是道路上的经纬度网格，而是固定位置的道路传感器。

In [ ]:
pd.DataFrame(raw[:12, :6], columns=[f'sensor_{i}' for i in range(6)])

上表只有前 12 个时刻和前 6 个节点。横向读是在同一时刻比较不同地点；纵向读是在同一地点观察速度变化。

**小练习 1：** 找出 `raw[0, 0]`、`raw[1, 0]` 和 `raw[0, 1]`，分别用一句自然语言解释。

In [ ]:
print('raw[0, 0] =', raw[0, 0])
print('raw[1, 0] =', raw[1, 0])
print('raw[0, 1] =', raw[0, 1])

## 2. 缺失值不是速度 0

项目把非有限值以及 `speed <= 0` 视为缺失。这样做沿用交通预测基准的常见约定：高速公路传感器中的 0 往往表示未上报，而不是车辆真的以 0 速度通过。

**先思考：** 如果把缺失的 0 当成真实速度计入 MAE，会让模型受到什么错误奖励或惩罚？

In [ ]:
observed_mask = np.isfinite(raw) & (raw > 0)
missing_count = (~observed_mask).sum()
print('总元素数:', raw.size)
print('缺失元素数:', missing_count)
print(f'缺失比例: {missing_count / raw.size:.4%}')
print('完全无缺失的传感器数:', np.all(observed_mask, axis=0).sum())

In [ ]:
missing_by_sensor = 1 - observed_mask.mean(axis=0)
worst = np.argsort(missing_by_sensor)[-10:][::-1]
pd.DataFrame({
    'sensor_index': worst,
    'missing_fraction': missing_by_sensor[worst],
})

`mask` 的作用是告诉损失函数和指标：只在真实观测位置比较预测与目标。缺失的输入仍要有一个数值占位；本项目在标准化后用 0 填充，含义是“训练集平均速度”，而不是原始速度 0。

## 3. 只使用训练期统计量进行标准化

标准化公式为：

$$z = \frac{x-\mu_{train}}{\sigma_{train}}$$

如果用完整数据计算均值和标准差，验证期和测试期的分布信息就提前进入了训练流程，这是一种不明显的数据泄漏。

In [ ]:
bundle = build_data_bundle(config.dataset)
train_end, val_end = bundle.split_points

print('train 时间范围: [0,', train_end, ')')
print('val   时间范围: [', train_end, ',', val_end, ')')
print('test  时间范围: [', val_end, ',', len(raw), ')')
print('scaler.mean:', bundle.scaler.mean)
print('scaler.std :', bundle.scaler.std)

In [ ]:
train_observed = raw[:train_end][observed_mask[:train_end]]
all_observed = raw[observed_mask]

print('手工训练集均值:', train_observed.mean())
print('代码中的均值  :', bundle.scaler.mean)
print('完整数据均值  :', all_observed.mean())
assert np.isclose(train_observed.mean(), bundle.scaler.mean)
print('✓ scaler 确实只使用训练期观测值')

**小练习 2：** 比较训练集均值和完整数据均值。二者即使非常接近，也不能使用完整数据均值。实验规范由信息是否可用决定，而不是由数值差异大小决定。

## 4. 亲手构造一个 `12 → 12` 时间窗口

选择预测起点 `t` 后：

- 输入是 `raw[t-12:t]`；
- 目标是 `raw[t:t+12]`；
- Python 切片右端不包含，因此输入与目标没有重叠。

**先思考：** 第一个合法的 `t` 是多少？若 `t=12`，输入最后一个时刻和目标第一个时刻分别是多少？

In [ ]:
t = config.dataset.input_steps
manual_x = raw[t - config.dataset.input_steps:t]
manual_y = raw[t:t + config.dataset.output_steps]

print('预测起点 t:', t)
print('输入时刻:', list(range(t - 12, t)))
print('目标时刻:', list(range(t, t + 12)))
print('manual_x shape:', manual_x.shape)
print('manual_y shape:', manual_y.shape)

In [ ]:
x, y, mask = bundle.train[0]
print('TrafficDataset 返回的单样本形状：')
print('x   :', tuple(x.shape), '= [T_in, N, F]')
print('y   :', tuple(y.shape), '= [T_out, N]')
print('mask:', tuple(mask.shape), '= [T_out, N]')

assert x.shape == (12, raw.shape[1], 1)
assert y.shape == (12, raw.shape[1])
assert mask.shape == y.shape

注意 `x` 已经标准化，所以不能直接与 `manual_x` 比较。将它逆标准化后再检查。

In [ ]:
x_restored = bundle.scaler.inverse_transform(x[..., 0]).numpy()
positions = observed_mask[:12]
print('观测位置最大还原误差:', np.abs(x_restored[positions] - manual_x[positions]).max())
assert np.allclose(x_restored[positions], manual_x[positions], atol=1e-5)
print('✓ 第一个 Dataset 样本就是手工切出的第一个窗口')

## 5. DataLoader 为什么多出一个 batch 维度

模型通常同时处理多个窗口。DataLoader 将多个单样本堆叠起来，最前面增加 batch 维。

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(bundle.train, batch_size=4, shuffle=False)
batch_x, batch_y, batch_mask = next(iter(loader))
print('batch_x   :', tuple(batch_x.shape), '= [B,T_in,N,F]')
print('batch_y   :', tuple(batch_y.shape), '= [B,T_out,N]')
print('batch_mask:', tuple(batch_mask.shape))
assert batch_x.shape == (4, 12, raw.shape[1], 1)

这就是项目接口中 `[B,T_in,N,F]` 的来源：

- `B=4`：四个预测样本；
- `T_in=12`：过去 60 分钟；
- `N=207`：207 个传感器；
- `F=1`：每个节点目前只有速度这一种特征。

## 6. 时间切分与泄漏检查

项目按样本的**第一个目标时刻**分配 split。验证样本可以使用训练边界之前的历史作为输入，因为真实预测时这些过去信息本来就是可用的；但训练集和验证集的目标时刻绝不能重叠。

**先思考：** 最后一个训练样本预测的最后时刻，应该小于、等于还是大于第一个验证样本的预测起点？

In [ ]:
train_last_start = int(bundle.train.indices[-1])
val_first_start = int(bundle.val.indices[0])
test_first_start = int(bundle.test.indices[0])
output_steps = config.dataset.output_steps

print('最后一个训练预测窗口:', list(range(train_last_start, train_last_start + output_steps)))
print('第一个验证预测窗口:', list(range(val_first_start, val_first_start + output_steps)))
print('第一个测试预测起点:', test_first_start)

assert train_last_start + output_steps <= val_first_start
assert int(bundle.val.indices[-1]) + output_steps <= test_first_start
print('✓ train/val/test 的目标时间没有重叠')

In [ ]:
print('窗口数量')
print('train:', len(bundle.train))
print('val  :', len(bundle.val))
print('test :', len(bundle.test))
print('总计 :', len(bundle.train) + len(bundle.val) + len(bundle.test))

边界附近会有少量没有被任何 split 使用的预测起点。这是有意的：若保留它们，长度为 12 的目标窗口就会跨越 split 边界。少用几个样本换来清晰、严格的评测协议是值得的。

## 7. 看见交通的周期性

现在暂时离开张量形状，看看数据是否具有模型可以学习的时间规律。先绘制一个传感器连续七天的速度。

In [ ]:
sensor = 0
week_steps = 7 * config.dataset.steps_per_day
series = np.where(observed_mask[:week_steps, sensor], raw[:week_steps, sensor], np.nan)
hours = np.arange(week_steps) * 5 / 60

plt.figure(figsize=(14, 3.5))
plt.plot(hours, series, linewidth=0.8)
for day in range(1, 7):
    plt.axvline(day * 24, color='gray', alpha=0.25)
plt.xlabel('Hour from start')
plt.ylabel('Speed')
plt.title(f'Sensor {sensor}: first seven days')
plt.tight_layout()
plt.show()

In [ ]:
slots = np.arange(len(raw)) % config.dataset.steps_per_day
daily_profile = np.array([
    np.nanmean(np.where(observed_mask[slots == slot], raw[slots == slot], np.nan))
    for slot in range(config.dataset.steps_per_day)
])

plt.figure(figsize=(9, 3.5))
plt.plot(np.arange(288) * 5 / 60, daily_profile)
plt.xlabel('Hour of day')
plt.ylabel('Mean speed across days and sensors')
plt.title('METR-LA average daily profile')
plt.xticks(range(0, 25, 3))
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

**小练习 3：**

1. 图中是否能看到早晚高峰？速度下降大约发生在哪些小时？
2. 将 `sensor=0` 改成缺失比例最高的传感器，图像有什么变化？
3. 只画周末的 daily profile，并与工作日比较。不要急着写完美代码，先用布尔索引实现。

## 8. 少样本的“3 天”究竟限制了什么

少样本实验不会缩短验证集和测试集，也不会重新计算一个更容易的测试集。它只限制目标城市允许用于优化参数的训练期长度。这样 scratch 和 transfer 才面对同一个测试问题。

In [ ]:
few_shot_bundle = build_data_bundle(config.dataset, few_shot_days=3)
print('完整训练窗口数:', len(bundle.train))
print('3 天训练窗口数:', len(few_shot_bundle.train))
print('完整/少样本验证窗口数:', len(bundle.val), len(few_shot_bundle.val))
print('完整/少样本测试窗口数:', len(bundle.test), len(few_shot_bundle.test))
assert len(bundle.val) == len(few_shot_bundle.val)
assert len(bundle.test) == len(few_shot_bundle.test)

请注意：3 天共有 864 个时间点，但可用训练窗口略少，因为第一个样本需要 12 步历史，最后一个完整目标还需要 12 步未来。所谓“3 天数据”不是“恰好 864 个独立样本”。

## 9. 本课验收

在继续第二课之前，请不用查看上文，尝试回答：

1. `raw.shape == [T,N]` 中的 `T` 和 `N` 分别是什么？
2. 为什么单个 `x` 是 `[12,N,1]`，而模型收到的是 `[B,12,N,1]`？
3. 为什么不能用完整数据计算标准化均值？
4. 验证样本使用训练边界前的历史输入，为什么不属于泄漏？
5. `mask=False` 的目标为什么不能参与 MAE？
6. 3 天少样本实验改变了哪些数据，哪些数据保持不变？

最后运行项目的数据测试：

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_data.py'],
    cwd=ROOT, text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, '数据测试未通过，请把上面的错误发给我'

## 10. 请带回来的结果

运行完后，不需要发整个 notebook。请记录并告诉我：

- `raw.shape` 和大约包含的天数；
- 缺失比例；
- train/val/test 窗口数量；
- 3 天训练窗口数量；
- daily profile 中速度最低的大致时段；
- 六个验收问题中最不确定的一个。

下一课会从这些结果出发，手算 Historical Average、MAE、RMSE 和 Masked MAPE，再解释你已经获得的 STGCN 指标。